In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import GridSearchCV,train_test_split,cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

In [15]:
!pip install pyxlsb

In [16]:
!pip install xlrd

In [17]:
import pandas as pd
import numpy as np
import warnings

# Suppress the specific pandas warnings
warnings.filterwarnings('ignore', message='invalid value encountered in greater')
warnings.filterwarnings('ignore', message='invalid value encountered in less')

# Or suppress all RuntimeWarnings from pandas formatting
warnings.filterwarnings('ignore', category=RuntimeWarning, module='pandas.io.formats.format')

# Now read your data
df = pd.read_excel('rawData.xlsx')
df = df[df['Sign Type Broad Category'] == 'Blade Sign']
df = df.reset_index(drop=True)


print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Check for data quality issues that might cause these warnings
print("\n" + "="*50)
print("DATA QUALITY CHECK:")

# Check for missing values
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing values:")
missing_cols = df.isnull().sum()
for col in missing_cols[missing_cols > 0].index:
    print(f"  - {col}: {missing_cols[col]} missing")

# Check data types
print(f"\nData types:")
for col, dtype in df.dtypes.items():
    print(f"  - {col}: {dtype}")

# Check for mixed data types in numeric columns
print(f"\nChecking for mixed data types:")
for col in df.columns:
    if df[col].dtype == 'object':  # String columns might contain mixed types
        # Try to identify if it should be numeric
        sample_values = df[col].dropna().astype(str).str.strip()
        if len(sample_values) > 0:
            # Check if values look numeric
            numeric_pattern = sample_values.str.match(r'^-?\d+\.?\d*$')
            if numeric_pattern.any():
                numeric_count = numeric_pattern.sum()
                total_count = len(sample_values)
                if numeric_count > total_count * 0.5:  # More than 50% numeric
                    print(f"  - {col}: Appears to be numeric but stored as object ({numeric_count}/{total_count} numeric)")

# Safe display function that handles problematic data
def safe_display(df, n_rows=5):
    """Display dataframe without triggering formatting warnings"""
    try:
        # Create a copy for display
        display_df = df.head(n_rows).copy()
        
        # Replace problematic values for display
        for col in display_df.columns:
            if display_df[col].dtype in ['float64', 'int64']:
                # Replace inf and -inf with string representations
                display_df[col] = display_df[col].replace([np.inf, -np.inf], ['inf', '-inf'])
        
        return display_df
    except Exception as e:
        print(f"Display error: {e}")
        return df.head(n_rows)

print(f"\n" + "="*50)
print("FIRST FEW ROWS (safe display):")
display_data = safe_display(df)
print(display_data)

# Clean up numeric columns if needed
print(f"\n" + "="*50)
print("CLEANING NUMERIC COLUMNS:")

numeric_cols = []
for col in df.columns:
    if 'price' in col.lower() or 'cost' in col.lower() or 'amount' in col.lower() or 'width' in col.lower() or 'height' in col.lower() or 'area' in col.lower():
        numeric_cols.append(col)

if numeric_cols:
    print(f"Found potential numeric columns: {numeric_cols}")
    
    for col in numeric_cols:
        if col in df.columns:
            print(f"\nCleaning column: {col}")
            original_type = df[col].dtype
            
            try:
                # Convert to numeric, coercing errors to NaN
                df[col] = pd.to_numeric(df[col], errors='coerce')
                print(f"  - Converted from {original_type} to {df[col].dtype}")
                print(f"  - NaN values after conversion: {df[col].isnull().sum()}")
                
            except Exception as e:
                print(f"  - Could not convert {col}: {e}")

# Final summary
print(f"\n" + "="*50)
print("FINAL DATA SUMMARY:")
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"Total missing values: {df.isnull().sum().sum()}")

# Display basic statistics for numeric columns
numeric_columns = df.select_dtypes(include=[np.number]).columns
if len(numeric_columns) > 0:
    print(f"\nNumeric columns summary:")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        print(df[numeric_columns].describe())

print("\nWarnings should now be suppressed!")
print("\nYour dataframe is ready to use: 'df'")

Data loaded successfully!
Shape: (538, 19)
Columns: ['Order ID', 'Order Date', 'Order Added in Month Tab', 'Account Name', 'Sign Type', 'Sign Type Broad Category', 'Month (AT)', 'Sign Width (in)', 'Sign Height (in)', 'Selling Price (USD)', 'Withdrawal Amount (USD)', 'Project Name', 'Status', 'Production Line', 'BOM - Material Cost (PKR) - Calculated By Ali Hassan', '📙 BOM - Production Cost (USD)', '📙 BOM - Shipping Cost (USD)', 'Sign Area (sq.ft)', 'Length of Curve (m)']

DATA QUALITY CHECK:
Total missing values: 1396
Columns with missing values:
  - Sign Width (in): 2 missing
  - Sign Height (in): 2 missing
  - Withdrawal Amount (USD): 538 missing
  - Status: 155 missing
  - BOM - Material Cost (PKR) - Calculated By Ali Hassan: 16 missing
  - 📙 BOM - Shipping Cost (USD): 145 missing
  - Length of Curve (m): 538 missing

Data types:
  - Order ID: object
  - Order Date: datetime64[ns]
  - Order Added in Month Tab: datetime64[ns]
  - Account Name: object
  - Sign Type: object
  - Sign Ty

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 19 columns):
 #   Column                                                Non-Null Count  Dtype         
---  ------                                                --------------  -----         
 0   Order ID                                              538 non-null    object        
 1   Order Date                                            538 non-null    datetime64[ns]
 2   Order Added in Month Tab                              538 non-null    datetime64[ns]
 3   Account Name                                          538 non-null    object        
 4   Sign Type                                             538 non-null    object        
 5   Sign Type Broad Category                              538 non-null    object        
 6   Month (AT)                                            538 non-null    datetime64[ns]
 7   Sign Width (in)                                       536 non-null    float64   

In [19]:
print(f'The number of rows are {df.shape[0]} and columns are {df.shape[1]}')

The number of rows are 538 and columns are 19


In [20]:
df.columns

Index(['Order ID', 'Order Date', 'Order Added in Month Tab', 'Account Name',
       'Sign Type', 'Sign Type Broad Category', 'Month (AT)',
       'Sign Width (in)', 'Sign Height (in)', 'Selling Price (USD)',
       'Withdrawal Amount (USD)', 'Project Name', 'Status', 'Production Line',
       'BOM - Material Cost (PKR) - Calculated By Ali Hassan',
       '📙 BOM - Production Cost (USD)', '📙 BOM - Shipping Cost (USD)',
       'Sign Area (sq.ft)', 'Length of Curve (m)'],
      dtype='object')

In [21]:
df.head(5)

,Order ID,Order Date,Order Added in Month Tab,Account Name,Sign Type,Sign Type Broad Category,Month (AT),Sign Width (in),Sign Height (in),Selling Price (USD),Withdrawal Amount (USD),Project Name,Status,Production Line,BOM - Material Cost (PKR) - Calculated By Ali Hassan,📙 BOM - Production Cost (USD),📙 BOM - Shipping Cost (USD),Sign Area (sq.ft),Length of Curve (m)
0,BS-ET-7835,2024-12-31,2024-12-30,COMUNITYTreasures,Blade Sign,Blade Sign,2025-01-01,24.0,24.0,340.0,NaN,ETSY Project,Shipped,Business Sign,NaN,0.00,125.45,4,NaN
1,BS-SM-7826 A,2024-12-30,2025-01-01,Signmakerz-Ads,3D Blade Sign,Blade Sign,2025-01-01,30.0,30.0,1248.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,380.90,241.30,6,NaN
2,BS-SM-7826 B,2024-12-30,2025-01-01,Signmakerz-Ads,3D Blade Sign,Blade Sign,2025-01-01,30.0,30.0,1248.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,380.90,241.30,6,NaN
3,BS-SM-7842,2024-12-31,2025-01-02,Signmakerz-Ads,Blade Sign,Blade Sign,2025-01-01,24.0,24.0,248.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,108.76,92.73,4,NaN
4,BS-SM-7851,2025-01-01,2025-01-03,Signmakerz-Ads,Blade Sign,Blade Sign,2025-01-01,30.0,30.0,378.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,91.31,165.79,6,NaN


In [22]:
df.isnull().sum()

Order ID                                                  0
Order Date                                                0
Order Added in Month Tab                                  0
Account Name                                              0
Sign Type                                                 0
Sign Type Broad Category                                  0
Month (AT)                                                0
Sign Width (in)                                           2
Sign Height (in)                                          2
Selling Price (USD)                                       0
Withdrawal Amount (USD)                                 538
Project Name                                              0
Status                                                  155
Production Line                                           0
BOM - Material Cost (PKR) - Calculated By Ali Hassan    538
📙 BOM - Production Cost (USD)                             0
📙 BOM - Shipping Cost (USD)             

In [23]:
df.drop_duplicates(inplace=True)

In [24]:
df.describe(include='all')

,Order ID,Order Date,Order Added in Month Tab,Account Name,Sign Type,Sign Type Broad Category,Month (AT),Sign Width (in),Sign Height (in),Selling Price (USD),Withdrawal Amount (USD),Project Name,Status,Production Line,BOM - Material Cost (PKR) - Calculated By Ali Hassan,📙 BOM - Production Cost (USD),📙 BOM - Shipping Cost (USD),Sign Area (sq.ft),Length of Curve (m)
count,538,538,538,538,538,538,538,536.000000,536.000000,538.000000,0.0,538,383,538,0.0,538.000000,393.000000,538.000000,0.0
unique,538,NaN,NaN,46,5,1,NaN,NaN,NaN,NaN,NaN,5,13,1,NaN,NaN,NaN,NaN,NaN
top,BS-ET-11443,NaN,NaN,HudsonByDino,Blade Sign,Blade Sign,NaN,NaN,NaN,NaN,NaN,ETSY Project,Shipped,Business Sign,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,73,461,538,NaN,NaN,NaN,NaN,NaN,452,348,538,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-04-28 20:33:54.200743680,2025-05-03 21:24:45.501858816,NaN,NaN,NaN,2025-04-18 10:58:26.319702528,20.707836,19.067351,339.624535,NaN,NaN,NaN,NaN,NaN,69.699089,93.559415,3.020446,NaN
min,NaN,2023-05-23 00:00:00,2023-05-23 00:00:00,NaN,NaN,NaN,2023-05-01 00:00:00,3.000000,2.500000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN
25%,NaN,2025-03-03 00:00:00,2025-03-08 00:00:00,NaN,NaN,NaN,2025-03-01 00:00:00,14.000000,12.000000,199.000000,NaN,NaN,NaN,NaN,NaN,28.015000,45.500000,1.000000,NaN
50%,NaN,2025-04-23 00:00:00,2025-04-27 00:00:00,NaN,NaN,NaN,2025-04-01 00:00:00,20.000000,18.000000,260.000000,NaN,NaN,NaN,NaN,NaN,48.910000,72.540000,2.000000,NaN
75%,NaN,2025-07-11 12:00:00,2025-07-15 00:00:00,NaN,NaN,NaN,2025-07-01 00:00:00,24.000000,24.000000,379.500000,NaN,NaN,NaN,NaN,NaN,84.375000,107.990000,4.000000,NaN
max,NaN,2025-08-28 00:00:00,2025-08-30 00:00:00,NaN,NaN,NaN,2025-08-01 00:00:00,60.000000,63.000000,3384.000000,NaN,NaN,NaN,NaN,NaN,610.770000,487.180000,18.000000,NaN


In [25]:
df['depth'] = 1

In [26]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from skopt import BayesSearchCV
import xgboost as xgb
import lightgbm as lgb
import pickle
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# =====================
# SETUP
# =====================

SIGN_TYPE = "blade_sign"
MODEL_DIR = os.path.join("model", SIGN_TYPE)
os.makedirs(MODEL_DIR, exist_ok=True)


# Rename columns
df_clean = df.copy()
df_clean = df_clean.rename(columns={
    'Sign Width (in)': 'width',
    'Sign Height (in)': 'height',
    'Depth': 'depth',
    '📙 BOM - Shipping Cost (USD)' : 'shipping_cost',
    'Sign Area (sq.ft)' : 'Sign Area(in)'
})

print("Renamed columns successfully!")

# =====================
# DATA PREP
# =====================
feature_columns = ['width', 'height', 'depth', 'Sign Area(in)', 'shipping_cost']
data_for_imputation = df_clean[feature_columns].copy()

print("\nApplying KNN Imputation...")
knn_imputer = KNNImputer(n_neighbors=4)
data_imputed = knn_imputer.fit_transform(data_for_imputation)
df_imputed = pd.DataFrame(data_imputed, columns=feature_columns, index=df_clean.index)

# Features / Target
X = df_imputed[['width', 'height', 'depth', 'Sign Area(in)']].copy()
y = df_imputed['shipping_cost'].copy()

# Scale + Split
scaler = MaxAbsScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# =====================
# MODEL TRAINING
# =====================
models_results = {}

def train_and_eval(name, estimator, param_grid, n_iter=50):
    """Train model with BayesSearchCV safely"""
    try:
        print(f"\n{name} with Bayesian Optimization:")
        search = BayesSearchCV(
            estimator=estimator,
            search_spaces=param_grid,
            n_iter=n_iter,
            cv=5,
            scoring='r2',
            random_state=42,
            n_jobs=-1
        )
        search.fit(X_train, y_train)
        y_pred = search.best_estimator_.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        print(f"{name} Test Metrics → MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")
        models_results[name] = {
            'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'model': search.best_estimator_
        }
    except Exception as e:
        print(f"⚠️ {name} failed: {e}")

# =====================
# TRAIN ALL MODELS
# =====================
train_and_eval("Random Forest", RandomForestRegressor(random_state=42), {
    'n_estimators': (10, 200), 'max_depth': (3, 20),
    'min_samples_split': (2, 15), 'min_samples_leaf': (1, 10),
    'max_features': ['sqrt', 'log2', None]
})

train_and_eval("Gradient Boosting", GradientBoostingRegressor(random_state=42), {
    'n_estimators': (50, 300), 'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3), 'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10), 'subsample': (0.8, 1.0)
})

train_and_eval("SVR", SVR(kernel='rbf'), {
    'C': (1, 1000), 'gamma': (0.001, 1), 'epsilon': (0.01, 1)
})

train_and_eval("Decision Tree", DecisionTreeRegressor(random_state=42), {
    'max_depth': (3, 20), 'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10), 'max_features': ['sqrt', 'log2', None]
})

train_and_eval("XGBoost", xgb.XGBRegressor(random_state=42, eval_metric='rmse'), {
    'n_estimators': (50, 300), 'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3), 'subsample': (0.8, 1.0),
    'colsample_bytree': (0.8, 1.0), 'reg_alpha': (0, 1), 'reg_lambda': (0, 1)
})

train_and_eval("LightGBM", lgb.LGBMRegressor(random_state=42, verbose=-1), {
    'n_estimators': (50, 300), 'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3), 'subsample': (0.8, 1.0),
    'colsample_bytree': (0.8, 1.0), 'reg_alpha': (0, 1),
    'reg_lambda': (0, 1), 'num_leaves': (10, 100)
})

train_and_eval("Extra Trees", ExtraTreesRegressor(random_state=42), {
    'n_estimators': (10, 200), 'max_depth': (3, 20),
    'min_samples_split': (2, 15), 'min_samples_leaf': (1, 10),
    'max_features': ['sqrt', 'log2', None]
})

train_and_eval("Ridge", Ridge(random_state=42), {
    'alpha': (0.1, 100)
}, n_iter=30)

train_and_eval("ElasticNet", ElasticNet(random_state=42), {
    'alpha': (0.1, 10), 'l1_ratio': (0.1, 0.9)
}, n_iter=30)

# =====================
# BEST MODEL SELECTION
# =====================
if models_results:
    best_model_name = max(models_results.keys(), key=lambda x: models_results[x]['R2'])
    best_model = models_results[best_model_name]['model']
    print(f"\n✅ Best Model: {best_model_name} (R² = {models_results[best_model_name]['R2']:.3f})")
else:
    best_model_name, best_model = None, None
    print("\n❌ No models trained successfully!")

# =====================
# SAVE MODELS
# =====================
if models_results:
    high_accuracy_models = {n: r for n, r in models_results.items() if r['R2'] > 0.7}
    to_save = high_accuracy_models if high_accuracy_models else dict(list(models_results.items())[:3])

    for model_name, results in to_save.items():
        joblib.dump(results['model'], f"{MODEL_DIR}/{model_name.replace(' ', '_').lower()}_model.joblib")
        with open(f"{MODEL_DIR}/{model_name.replace(' ', '_').lower()}_model.pkl", 'wb') as f:
            pickle.dump(results['model'], f)

    if best_model:
        joblib.dump(best_model, f"{MODEL_DIR}/best_model_{best_model_name.replace(' ', '_').lower()}.joblib")

    joblib.dump(scaler, f"{MODEL_DIR}/scaler.joblib")
    with open(f"{MODEL_DIR}/feature_names.pkl", 'wb') as f:
        pickle.dump(['width', 'height', 'depth', 'Selling Price (USD)', 'Sign Area (sq.ft)'], f)
    with open(f"{MODEL_DIR}/model_results.pkl", 'wb') as f:
        pickle.dump(models_results, f)

    print(f"\n📂 Saved models in {MODEL_DIR}/")
    for file in os.listdir(MODEL_DIR):
        print(" -", file)

print("\n🚀 Mission accomplished!")


Renamed columns successfully!

Applying KNN Imputation...
Train: (430, 4), Test: (108, 4)

Random Forest with Bayesian Optimization:
Random Forest Test Metrics → MAE: 28.71, RMSE: 42.77, R²: 0.478

Gradient Boosting with Bayesian Optimization:
Gradient Boosting Test Metrics → MAE: 28.37, RMSE: 41.27, R²: 0.514

SVR with Bayesian Optimization:
SVR Test Metrics → MAE: 29.52, RMSE: 43.76, R²: 0.453

Decision Tree with Bayesian Optimization:
Decision Tree Test Metrics → MAE: 31.02, RMSE: 45.22, R²: 0.416

XGBoost with Bayesian Optimization:
XGBoost Test Metrics → MAE: 30.93, RMSE: 43.10, R²: 0.470

LightGBM with Bayesian Optimization:
LightGBM Test Metrics → MAE: 28.81, RMSE: 40.79, R²: 0.525

Extra Trees with Bayesian Optimization:
Extra Trees Test Metrics → MAE: 29.13, RMSE: 42.27, R²: 0.490

Ridge with Bayesian Optimization:
Ridge Test Metrics → MAE: 33.97, RMSE: 42.58, R²: 0.483

ElasticNet with Bayesian Optimization:
ElasticNet Test Metrics → MAE: 34.20, RMSE: 42.97, R²: 0.473

✅ Best